# NT Autofix: Greek cleanup + Latvian enforcement + Exact count matching

In-place edits on chapter JSONs (like OT `4split_json_upgraded`):

1. **Strip Greek punctuation** — apply `raccs_common` to every `greek` field  
2. **Greek count enforcement** — ensure JSON has exactly the same number of Greek words as strongs (stripped diacritics comparison)  
3. **Fix latvian junk** — remove entries that are just punctuation (`,` etc.)  
4. **Latvian count enforcement** — ensure JSON has exactly the same Latvian words as lv65 (stripped diacritics comparison)  
5. **Remove extras** — first from leftovers, then from mappings  
6. **Add missing** — missing latvian words added to leftover_latvian

## 0 – Config & shared helpers

In [97]:
from pathlib import Path
import json, re, unicodedata, os
from collections import Counter, defaultdict
import pandas as pd

BASE_DIR = Path("bible")
DRY_RUN  = 1#False#True        # True = preview only, False = write files
#VERBOSE  = 1#False #True
VERBOSE_GK =1
VERBOSE_LV =1
VERBOSE_ALL =  VERBOSE_GK and VERBOSE_LV

LV_TRANSTBL = str.maketrans('āčēģīķļņōŗšūžĀČĒĢĪĶĻŅŌŖŠŪŽ',
                            'acegiklnorsuzACEGIKLNORSUZ')
# ── raccs_common: latvians and punctuation stripper ──────────────────────────────
def raccs_common(text):
    d = {
        ord('\u2019'): None,  # RIGHT SINGLE QUOTATION MARK  \u2019
        ord('\u2018'): None,  # LEFT SINGLE QUOTATION MARK   \u2018
        ord('\u201C'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('\u201D'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,
        ord(']'): None,
        ord('-'): None,
        ord("'"): None,
        ord('\u29FC'): None,  # \u29fc
        ord('\u29FD'): None,  # \u29fd
        ord('*'): None,
        ord('\u21D4'): None,  # \u21d4
        ord('\u3009'): None,  # \u3009
        ord('\u3008'): None,  # \u3008
        ord('\u203F'): None,  # \u203f
        ord('\u00AB'): None,  # \u00ab
        ord('\u00BB'): None,  # \u00bb
        ord('\u2039'): None,  # \u2039
        ord('\u203A'): None,  # \u203a
        ord('('): None,
        ord(')'): None,
        ord(';'): None,
        ord('{'): None,
        ord('}'): None,
        ord('.'): None,
        ord(','): None,
        ord('!'): None,
        ord('?'): None,
        ord(';'): None,
        ord(':'): None,
        ord('"'): None,
        ord(')'): None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
# nestrādā .... nooooooooooooooooooooooo
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        ord('᾽') : None,  # COLON
    }
    return unicodedata.normalize('NFC', text).translate(d).translate(LV_TRANSTBL)



# ── Greek / Latvian helpers ───────────────────────────────────────────────
def strip_greek_diacritics(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

LV_CHARS = "A-Za-z\u0101\u0113\u012b\u016b\u0123\u0137\u013c\u0146\u0161\u017e\u010d\u0100\u0112\u012a\u016a\u0122\u0136\u013b\u0145\u0160\u017d\u010c"
LV_REGEX = re.compile(f"[{LV_CHARS}]+")

def lv_tokens(text: str):
    return LV_REGEX.findall(unicodedata.normalize('NFC', text))

def strip_lv_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower())
                   if unicodedata.category(c) != 'Mn')

## 1 – Load reference datasets

In [8]:
strongs_df = pd.read_csv("strongs.csv")
lv65_df    = pd.read_csv("latvian_full65.csv")

# Pre-clean strongs forms with raccs_common so we compare apples to apples
strongs_df["form_clean"] = strongs_df["form"].astype(str).apply(raccs_common)

strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
lv_g      = lv65_df.groupby(["book", "chapter", "verse"])

print(f"Strongs: {len(strongs_df)} rows, LV65: {len(lv65_df)} rows")

Strongs: 138993 rows, LV65: 7956 rows


## 2 – Dense compact writer (same as OT pattern)

In [9]:
def write_dense_json(file_path, data):
    """
    Write chapter data back in OT-style dense compact format.
    data = list of verse dicts, each with greek_words + leftover_latvian.
    """
    def word_to_compact(w):
        gr  = json.dumps(w.get('greek', ''), ensure_ascii=False)
        lv  = json.dumps(w.get('latvian', []), ensure_ascii=False)
        return f'{{ "greek": {gr}, "latvian": {lv} }}'

    verse_blocks = []
    for verse in data:
        words = verse.get('greek_words', [])
        compact_words = ',\n      '.join(word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "greek_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )

    new_raw = '[\n' + ',\n'.join(verse_blocks) + '\n]'
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(new_raw)

## 3 – Autofix function

For each verse in a chapter JSON:

**Pass 1 – Greek cleanup + count enforcement**  
**Pass 2 – Latvian junk removal**  
**Pass 3 – Latvian count enforcement:**  
- Match using stripped diacritics to avoid reruns  
- **Prefix fan-out**: `ne-` + `viens` + `var` → matches ref `neviens` + `nevar`  
  (one prefix entry can compound with multiple roots)  
- **Suffix merge**: `root` + `-suffix` → compound  
- Add missing ref words to leftover_latvian  
- Remove extras: first from leftovers, then from mappings

In [138]:
def autofix_chapter(book_name, chapter_num, strongs_g, lv_g,
     verbose_gk=True, verbose_lv=True, verbose_all=True, 
     silence_gk_clean=False, silence_lv_edit=False,
     fix_latvian=True, dry_run=True):
    """
    In-place autofix for a single NT chapter JSON.
    Returns (had_fixes: bool, fix_log: list).

    Greek pass:
      - Builds an index mapping from dataset → JSON using stripped-diacritics comparison
      - Extra JSON words (not in dataset) are removed; their latvian mappings are salvaged
      - Missing dataset words are inserted at the correct position
      - Final greek_words exactly matches the dataset in order and count
      - All greek forms are cleaned via raccs_common

    Latvian pass:
      - Counts all latvian tokens (mappings + leftovers) vs lv65 dataset
      - If any mapping contains "ne-" or "ne", enables ne-compound awareness:
        dataset words like "netopiet" also accept stem "topiet" as a match
        (but each dataset word is matched at most once, either full or stem)
      - Missing words → added to leftover_latvian
      - Excess words → removed from leftovers first, then from mappings
    """
    json_path = BASE_DIR / book_name / f"{chapter_num}.json"
    if not json_path.exists():
        return False, []

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    fixes = []

    for vi, verse_data in enumerate(data):
        verse_num = vi + 1
        key = (book_name, chapter_num, verse_num)
        greek_words = verse_data.get('greek_words', [])
        leftover_latvian = verse_data.get('leftover_latvian', [])

        lv_text=[]
        ref_lv=[]
        if key in lv_g.groups:
            lv_text = lv_g.get_group(key).iloc[0]['text']
            ref_lv = lv_tokens(lv_text)
        hadNe = False
        # ═══════════════════════════════════════════════════════════════════
        # GREEK PASS: Align JSON greek words to dataset by index matching
        # ═══════════════════════════════════════════════════════════════════
        if key in strongs_g.groups:
            ref_df = strongs_g.get_group(key).sort_values('word')
            ref_forms = [str(ref_df.iloc[i]['form']) for i in range(len(ref_df))]
            ref_stripped = [strip_greek_diacritics(raccs_common(f)).strip().lower() for f in ref_forms]
            ref_count = len(ref_forms)

            json_stripped = [strip_greek_diacritics(raccs_common(gw.get('greek', ''))).strip().lower()
                            for gw in greek_words]
            json_count = len(greek_words)

            # --- Build index array: ref_map[ref_idx] = json_idx or -1 (missing) ---
            # Each ref word tries to find a matching json word (in order, not yet taken)
            ref_map = [-1] * ref_count       # -1 = not yet matched
            json_taken = [False] * json_count  # track which json entries are claimed

            # Greedy forward match preserving order:
            # Walk ref in order, for each ref word find the earliest untaken json match
            lastTakenTOP = 0
            for ri in range(ref_count):
                target = ref_stripped[ri]
                for ji in range(lastTakenTOP, json_count): 
                    if not json_taken[ji] and json_stripped[ji] == target:
                        if ji == lastTakenTOP+1:
                            lastTakenTOP = ji
                        ref_map[ri] = ji
                        json_taken[ji] = True
                        break

            # Any json entry not taken is "extra" → mark for removal, salvage latvian
            extras = [ji for ji in range(json_count) if not json_taken[ji]]
            salvaged_latvian = []
            if extras:
                for ji in extras:
                    removed_gw = greek_words[ji]
                    old_gr = removed_gw.get('greek', '')
                    salvaged_lv = removed_gw.get('latvian', [])
                    if salvaged_lv:
                        salvaged_latvian.extend(salvaged_lv)
                    if verbose_all or verbose_gk:
                        print(f"  ➖🇬🇷 v{verse_num} w{ji+1}: removed extra '{old_gr}' lv={salvaged_lv}")
                    fixes.append(('greek_extra_removed', key, len(extras)))

            # --- Rebuild greek_words in exact dataset order ---
            # cleaned form means word from dataset is stripmed whitespace AND removed raccs chars, also accents diatrics .
            new_greek_words = []
            for ri in range(ref_count):
                cleaned_form = raccs_common(ref_forms[ri]).strip()
                ji = ref_map[ri]
                if ji >= 0:
                    # Matched: take the json entry but fix the greek text
                    gw = dict(greek_words[ji])  # shallow copy
                    old_gr = gw.get('greek', '')
                    gw['greek'] = cleaned_form
                    if old_gr != cleaned_form:
                        if (verbose_all or verbose_gk) and not silence_gk_clean:
                            print(f"  ✏️🇬🇷 v{verse_num} ref#{ri+1}: '{old_gr}' → '{cleaned_form}'")
                        fixes.append(('greek_clean', key, ri + 1, old_gr, cleaned_form))
                    new_greek_words.append(gw)
                else:
                    # Missing in JSON: insert a new entry with no latvian mapping
                    if verbose_all or verbose_gk:
                        print(f"  ➕🇬🇷 v{verse_num} ref#{ri+1}: inserted missing '{cleaned_form}'")
                    new_greek_words.append({
                        'greek': cleaned_form,
                        'latvian': []
                    })
                    fixes.append(('greek_missing_inserted', key, ri + 1, cleaned_form))

            greek_words = new_greek_words

            # Add salvaged latvian to leftovers, merging ne- prefixes first
            # if lv verse not in lv dataset, drop the salvages
            if salvaged_latvian:
                if ref_lv:
                    hadNe, salvaged_latvian = _merge_ne_prefixes(salvaged_latvian, ref_lv)
                    leftover_latvian.extend(salvaged_latvian)
                    if verbose_all or verbose_lv:
                        print(f"  ♻️🇱🇻 v{verse_num} salvaged to leftovers: {salvaged_latvian}")

        else:
            # No strongs data for this verse — just clean greek forms in place
            if verbose_all or verbose_gk:
                print(f"  ⚠️🇬🇷 v{verse_num}: no strongs data, skipping index alignment, just cleaning forms")
            #now puts to greek accents!
            # for wi, gw in enumerate(greek_words):
            #     old_gr = gw.get('greek', '')
            #     new_gr = raccs_common(old_gr).strip()
            #     if new_gr != old_gr:
            #         if verbose_all or verbose_gk:
            #             print(f"  ✏️🇬🇷 v{verse_num} w{wi+1}: '{old_gr}' → '{new_gr}'")
            #         greek_words[wi]['greek'] = new_gr
            #         fixes.append(('greek_clean', key, wi + 1, old_gr, new_gr))

        if fix_latvian:
            if key in lv_g.groups and ref_lv:
                #first additional O(n^2) but that is needed for the spaces in middle..
                for i in range(len(greek_words)):
                    lv = greek_words[i].get('latvian', [])
                    newlv = []
                    for j in range (len(lv)):
                        stripped = lv[j].strip()
                        if(" " in stripped):
                            newlv.extend(stripped.split())
                            if verbose_all or verbose_lv:
                                print(f"  ➗🇱🇻 v{verse_num} w{i+1} mapping: split '{stripped}' into {stripped.split()}")
                            fixes.append(('latvian_space_split', key, len(extras)))
                        else:
                            newlv.append(stripped)
                    greek_words[i]['latvian'] = newlv

                _latvian_count_enforcement(
                    greek_words, leftover_latvian, ref_lv,
                    verse_num, key, fixes, verbose_all, verbose_gk, verbose_lv,
                    silence_lv_edit=silence_lv_edit, has_ne_prefix = hadNe
                ) 

        # Write back into data
        data[vi]['greek_words'] = greek_words
        data[vi]['leftover_latvian'] = leftover_latvian

    # ── Write if any fixes ───────────────────────────────────────────────
    if fixes and not dry_run:
        write_dense_json(json_path, data)
        if verbose_all or verbose_gk or verbose_lv:
            print(f"  ✅ {len(fixes)} fix(es) written to {json_path}")

    return bool(fixes), fixes


def _latvian_count_enforcement(greek_words, leftover_latvian, ref_lv,
                               verse_num, key, fixes, verbose_all, verbose_gk, verbose_lv,
                               silence_lv_edit=False, has_ne_prefix=False):
    """
    Enforce that latvian token counts in JSON match the lv65 dataset exactly.

    When "ne-" or "ne" appears in any mapping, enables ne-compound matching:
    a dataset word like "netopiet" can be matched by either "netopiet" or "topiet"
    (the stem after stripping the "ne" prefix). Each dataset word gets at most one
    match.
    """
    #first check the leftovers, because this removes the need to reiterate the mappings
    # in case when ne- is only inside the leftovers, not in mappings,
    #  we still want to trigger the ne-compound matching mode, 
    # because the presence of ne- in leftovers means that the verse is ne-aware, 
    # and we want to preserve the ne- compounds in mappings as well, 
    # and also allow the stem matches for the leftovers. 
    # so we need to check the leftovers for ne- presence as well, before we start matching.
    if not has_ne_prefix:
        for li, item in enumerate(leftover_latvian):
            if not item or item == '-':
                continue
            if not has_ne_prefix and (item[0]=='n' or item[0]=='N')  and len(item)>1 and (item[1] in ('e', 'E')) and len(item)<=3:
                has_ne_prefix = True
                break
    #first build counter dict for the ref.
    #then only go over the json, and remove exact matches fom dict counter
    #what is left is transformed into stripped counter
    ref_counts = {}
    for w in ref_lv:
        if not w or w == '-': continue 
        ##if len(w)==2 and w[0] in ('n', 'N') and  w[1]=='e':
            #skip the ne particles in the initial count
        ##    continue
        if w not in ref_counts:
            ref_counts[w]=1
        else:
            ref_counts[w]+=1

    #pass 1. direct match, mark as used.        
    untouchables_lamb_mark = []
    for gwi, gw in enumerate(greek_words):
        lvs = gw.get('latvian', [])
        untouchables_lamb_mark.append([False]*len(lvs))
        for li, item in enumerate(lvs):
            if not item or item == '-':
                continue
            if item in ref_counts:
                untouchables_lamb_mark[gwi][li]=True
                if ref_counts[item] == 1:
                    del ref_counts[item]
                else:
                    ref_counts[item] -=1
            else:
                # Detect ne- prefix markers
                if not has_ne_prefix and len(item)<4:
                    #nes
                    if (item[0]=='n' or item[0]=='N') and len(item)>1 and \
                         (item[1]=='e' or item[1]=='E')\
                        and item.lower() in ('ne-', 'ne'):
                        has_ne_prefix = True

    #pass 2. mark the soft matches, mark first found pair as used.
    #untouchables_lamb_mark mark which ones will not be removed in pass 3 (remove extra words not present in dataset)
    #now lets match softly the rest of mappings
    # --- Build unmapped ref counts (stripped ignore case) ---
    ref_stripped_counts = {}
    ne_ref_count=0
    for w, cnt in ref_counts.items():
        sw = raccs_common(w).strip().lower()
        if has_ne_prefix:
            #there is particle "ne" in Latvian too: ne vienu ne otru..
            #nesa nes nesen, will cause extra ne to preserve..
            if sw.startswith('ne') and len(sw)>=2:
                ne_ref_count+=1
        if sw not in ref_stripped_counts:
            ref_stripped_counts[sw] = cnt, [w]*cnt
        else:
            ref_stripped_counts[sw] = ref_stripped_counts[sw][0] + cnt, ref_stripped_counts[sw][1] + [w]*cnt

    ne_json_count = 0
    for gwi, gw in enumerate(greek_words):
        lvs = gw.get('latvian', [])
        for li, item in enumerate(lvs):
            if(untouchables_lamb_mark[gwi][li]): continue
            if not item or item == '-':
                continue
            cmp = raccs_common(item).strip().lower()
            if(len(cmp)==2 and len(item.strip())==3):
                sitem = item.strip()
                if (sitem[0]=='n' or sitem[0]=='N')  and (sitem[1] in ('e', 'E')) and sitem[2] =='-':
                    #THIS IS MARKER FOR NE-PREFIX, should not match to the ne particle!
                    cmp = cmp+"-"
            if  verse_num == 5:
                breakpoint()
            if cmp in ref_stripped_counts:
                untouchables_lamb_mark[gwi][li]=True
                count, forms = ref_stripped_counts[cmp]
                form0 = forms.pop(0) #remove the first form, which is now used, from the list of available forms for this stripped match
                greek_words[gwi]['latvian'][li] = form0 #replace with the form from dataset, which has diacritics and caps
                if (verbose_all or verbose_lv) and not silence_lv_edit:
                    print(f"  ✏️🇱🇻 v{verse_num} w{gwi+1}: '{item}' → '{form0}'")
                fixes.append(('lv_mapped_edit', key, item))
                if count == 1:
                    del ref_stripped_counts[cmp]
                else:
                    ref_stripped_counts[cmp] = count - 1, forms
                untouchables_lamb_mark[gwi][li]=True
                #exception here ensures data integrity.
                if ref_counts[form0] == 1:
                    del ref_counts[form0]
                else:
                    ref_counts[form0] -=1
                    #nes - matched.
            elif has_ne_prefix and (item[0]=='n' or item[0]=='N')  \
                and len(item)>1 and (item[1] in ('e', 'E')) and \
                    (True if len(item)==2 else item[2]=='-'):
                 #allow the ne prefix or ne particle to be there, unless they are more than matching ne prefixes/particles at dataset! then those extra must be removed
                 # now the case is lost, but that is the sacrifice required..
                ne_json_count +=1
                if(not ne_json_count>ne_ref_count):
                    untouchables_lamb_mark[gwi][li]=True
                else:
                    untouchables_lamb_mark[gwi][li]=False
                    if verbose_all or verbose_lv:
                        print(f"  ➖🇱🇻 v{verse_num} w{gwi+1}: removed extra '{item}' (excess ne-prefix/particle)")
                    fixes.append(('lv_mapped_strip', key, item))
            elif has_ne_prefix:
                # ne prefix match mode. single ne means multiple stems are allowed to match,
                #  but each dataset word can only be matched once, so we need to check the count 
                # and forms list for each match.
                refcopy = ref_stripped_counts.copy()
                #can raise RuntimeError: dictionary changed size during iteration if we try to modify the dict while iterating, so we need to iterate over a copy of the dict keys.
                for cmp in refcopy:
                    if cmp.startswith('ne') and len(cmp)>2 and cmp[2:]==raccs_common(item).strip().lower():
                        untouchables_lamb_mark[gwi][li]=True
                        count, forms = ref_stripped_counts[cmp]
                        form0 = forms.pop(0)
                        if(form0[2:]!=item):
                            greek_words[gwi]['latvian'][li] = form0 #replace with the form from dataset, which has diacritics and caps
                            if (verbose_all or verbose_lv) and not silence_lv_edit:
                                print(f"  ✏️🇱🇻 v{verse_num} w{gwi+1}: '{item}' → '{form0}' (ne-compound match)")
                            fixes.append(('lv_mapped_edit', key, item))
                        if count == 1:
                            del ref_stripped_counts[cmp]
                        else:
                             ref_stripped_counts[cmp] = count - 1, forms
                        #exception here ensures data integrity.
                        if ref_counts[form0] == 1:
                            del ref_counts[form0]
                        else:
                            ref_counts[form0] -=1
                        break

    # pass 3 EXODUS 12 !!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    for gwi, gw in enumerate(greek_words):
        lvs = gw.get('latvian', [])
        new_lvs = []
        for li, item in enumerate(lvs):
            if(untouchables_lamb_mark[gwi][li]): new_lvs.append(item)
            else:
                if verbose_all or verbose_lv:
                    print(f"  ➖🇱🇻 v{verse_num} w{gwi+1}: removed extra '{item}'")
                fixes.append(('lv_mapped_strip', key, item))
        greek_words[gwi]['latvian'] = new_lvs

    # pass 4.A then go with the leftovers.
    untouchables_LO = [False]*len(leftover_latvian) 
    for li, item in enumerate(leftover_latvian):
        if not item or item == '-':
            continue
        if item in ref_counts:
            untouchables_LO[li]=True
            stripped_item = raccs_common(item).strip().lower()
            if ref_counts[item] == 1:
                del ref_counts[item]
                number, items = ref_stripped_counts[stripped_item]
                if item in items:
                    items.remove(item)
                    if number == 1:
                        del ref_stripped_counts[stripped_item]
                    else:
                        ref_stripped_counts[stripped_item] = number-1, items
            else:
                ref_counts[item] -=1
                number, items = ref_stripped_counts[stripped_item]
                #match first item.
                items.remove(item)
                ref_stripped_counts[stripped_item] = number-1, items
                if ref_stripped_counts[stripped_item][0] == 0:
                    del ref_stripped_counts[stripped_item]
        else:
            cmp = raccs_common(item).strip().lower()
            if cmp in ref_stripped_counts:
                untouchables_LO[li]=True
                count, forms = ref_stripped_counts[cmp]
                form0 = forms.pop(0)
                #HERE REF_stripped is used also as initial array, so it can be they match already
                if(form0!=item):
                    leftover_latvian[li] = form0 #replace with the form from dataset, which has diacritics and caps
                    if (verbose_all or verbose_lv) and not silence_lv_edit:
                        print(f"  ✏️🇱🇻 v{verse_num} leftover #{li+1}: '{item}' → '{form0}'")
                    fixes.append(('lv_leftover_edit', key, [item]))

                if count == 1:
                    del ref_stripped_counts[cmp]
                    del ref_counts[form0]
                else:
                    ref_stripped_counts[cmp] = count - 1, forms
                    ref_counts[form0] -=1
            elif has_ne_prefix and (item[0]=='n' or item[0]=='N')  \
                and len(item)>1 and (item[1] in ('e', 'E')) and \
                    (True if len(item)==2 else item[2]=='-'):
                #allow the ne prefix or ne particle to be there, unless they are more than matching ne prefixes/particles at dataset! then those extra must be removed
                # now the case is lost, but that is the sacrifice required..
                ne_json_count +=1
                if(not ne_json_count>ne_ref_count):
                    untouchables_LO[li]=True
                else:
                    untouchables_LO[li]=False
                    if verbose_all or verbose_lv:
                        print(f"  ➖🇱🇻 v{verse_num} leftover #{li+1}: removed extra '{item}' (excess ne-prefix/particle)")
                    fixes.append(('lv_leftover_strip', key, item))
            elif has_ne_prefix:
                refcopy = ref_stripped_counts.copy()
                for cmp in refcopy:
                    if cmp.startswith('ne') and len(cmp)>2 and cmp[2:]==raccs_common(item).strip().lower():
                        untouchables_LO[li]=True
                        count, forms = ref_stripped_counts[cmp]
                        form0 = forms.pop(0)
                        if(form0[2:]!=item):
                            leftover_latvian[li] = form0 #replace with the form from dataset, which has diacritics and caps
                            if (verbose_all or verbose_lv) and not silence_lv_edit:
                                #if(li==5):
                                #    print([hex(ord(c)) for c in item])
                                #    print([hex(ord(c)) for c in form0])
                                print(f"  ✏️🇱🇻 v{verse_num} leftover #{li+1}: '{item}' → '{form0}' (ne-compound match)")
                            fixes.append(('lv_leftover_edit', key, [item]))
                        if count == 1:
                            del ref_stripped_counts[cmp]
                            del ref_counts[form0]
                        else:
                            ref_stripped_counts[cmp] = count - 1, forms
                            ref_counts[form0] -=1             
                        break
    # pass 4.B EXODUS 12 !!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    new_leftovers = []
    for li, item in enumerate(leftover_latvian):
        if(untouchables_LO[li]): new_leftovers.append(item)
        else:
            if verbose_all or verbose_lv:
                print(f"  ➖🇱🇻 v{verse_num} leftover #{li+1}: removed extra '{item}'")
            fixes.append(('lv_leftover_strip', key, item))
    leftover_latvian[:] = new_leftovers

    # pass 5. then add the missing ones from ref_counts to leftovers, no matter if ne or not, because if they were in mappings, they would be matched already, and if they were in leftovers, they would be matched in pass 4.
    for w, count in ref_counts.items():
        for _ in range(count):
            if verbose_all or verbose_lv:
                print(f"  ➕🇱🇻 v{verse_num} missing leftover: '{w}'")
            leftover_latvian.append(w)
            fixes.append(('lv_leftover_add', key, [w]))



def _merge_ne_prefixes(tokens, ref_lv):
    """
   trigger ne mode and return single ne- if any token has ne- or ne, and if there is at least 1 ne- or ne in the dataset. also return the modified tokens with merged ne- compounds if found.

    """
    if not tokens:
        return False, tokens

    neMatchMode = False
    for i, tok in enumerate(tokens):
        stripped = tok.strip()
        if stripped.lower() in ('ne-', 'ne'):
            neMatchMode = True
            break  
    # skip nē which is a valid Latvian word, and also can be part of other words like "nēģis" etc.
    # well nehemija will trigger this, but that is the sacrifice required to fix the ne- compounds. we can not just look for ne- prefix, because some of them are written as separate ne particle, and we want to merge those too.
    if neMatchMode:
        neMatchMode = False
        for wr in ref_lv:
            #w = wr.strip().lower()
            if len(wr)>2:
                if((wr[0]=='n' or wr[0]=='N' ) and( wr[1]=='e' or wr[1]=='E')):
                    neMatchMode = True
                    break
    # iff there is at least 1 ne in tokens, AND at leas 1 ne in dataset, keep single ne- at tokens. no more logic needed.
    #no point to optimize to len first, then char match, so kiss.
    #besides, later counts will adjust, so even add the duplicates for now!
    #2 tasks here: 1) remove alien words 2)merge ne if found
    soft_match_tokens = [unicodedata.normalize('NFC', w.strip().lower()) for w in tokens]
     # 1)                    
    # there is no single character word in Latvian, "a" is russian and not used in literal lang ("a ko tu domā")
    result = [w for w in soft_match_tokens if (len(w)>3) or\
                                 (len(w) == 2 and w!='ne') or (len(w)==3 and w != 'ne-') ]
    if neMatchMode: result.append('ne-')
        
    return neMatchMode, result


## 4 – Discover & run on all chapters

In [54]:
def discover_json_chapters(base_dir):
    """Return dict: book_name -> sorted list of chapter numbers with .json files."""
    result = {}
    for book_dir in sorted(base_dir.iterdir()):
        if not book_dir.is_dir():
            continue
        chapters = sorted(
            int(f.stem) for f in book_dir.iterdir()
            if f.is_file() and f.suffix == '.json' and f.stem.isdigit()
        )
        if chapters:
            result[book_dir.name] = chapters
    return result

book_chapters = discover_json_chapters(BASE_DIR)
total_ch = sum(len(v) for v in book_chapters.values())
print(f"Found {len(book_chapters)} books, {total_ch} chapter JSONs")

Found 27 books, 260 chapter JSONs


In [139]:
# ── Run autofix on all chapters ──────────────────────────────────────────
all_fixes = []
files_changed = 0
VERBOSE_GK = 0
VERBOSE_LV = 1
VERBOSE_ALL = 0
SILENCE_GK_CLEAN =1
SILENCE_LV_EDIT =1
FIX_LATVIAN = 1
DRY_RUN=1

breakpoint()
for book_name, chapters in book_chapters.items():
    for ch_num in chapters:
        #print(f"Processing {book_name} {ch_num}...")
        #if ch_num !=7 or book_name != '1_corinthians': continue #TEMP
        had_fixes, fix_log = autofix_chapter(
            book_name, ch_num, strongs_g, lv_g,
            verbose_gk=VERBOSE_GK, verbose_lv=VERBOSE_LV, verbose_all=VERBOSE_ALL,
            silence_gk_clean=SILENCE_GK_CLEAN, silence_lv_edit=SILENCE_LV_EDIT,
            fix_latvian=FIX_LATVIAN,
             dry_run=DRY_RUN
        )
        if had_fixes:
            files_changed += 1
            all_fixes.extend(fix_log)

action = 'Would fix' if DRY_RUN else 'Fixed'
print(f"\n{'='*60}")
print(f"{action} {files_changed} file(s), {len(all_fixes)} total fix(es).")
if DRY_RUN:
    print("Set DRY_RUN = False and re-run to write changes.")
fix_frame = pd.DataFrame(all_fixes)

  ➕🇱🇻 v9 missing leftover: 'ne'
  ➕🇱🇻 v8 missing leftover: 'ne'
  ➕🇱🇻 v10 missing leftover: 'ne'
  ➕🇱🇻 v51 missing leftover: 'ne'
  ➗🇱🇻 v19 w4 mapping: split '- apslāpējiet' into ['-', 'apslāpējiet']
  ➖🇱🇻 v19 w4: removed extra '-'
  ➗🇱🇻 v6 w6 mapping: split '- būtu' into ['-', 'būtu']
  ➖🇱🇻 v6 w6: removed extra '-'
  ➗🇱🇻 v21 w10 mapping: split 'tiks izglābts' into ['tiks', 'izglābts']
  ➗🇱🇻 v25 w20 mapping: split '- šaubītos' into ['-', 'šaubītos']
  ➖🇱🇻 v25 w20: removed extra '-'
  ➗🇱🇻 v43 w24 mapping: split 'Bābeles -' into ['Bābeles', '-']
  ➖🇱🇻 v43 w24: removed extra '-'
  ➗🇱🇻 v9 w12 mapping: split '- zinādams' into ['-', 'zinādams']
  ➖🇱🇻 v9 w12: removed extra '-'
  ➗🇱🇻 v1 w1 mapping: split 'tur bija' into ['tur', 'bija']
  ➗🇱🇻 v1 w10 mapping: split 'valdības vīrs' into ['valdības', 'vīrs']
  ➗🇱🇻 v34 w1 mapping: split 'Jo ko' into ['Jo', 'ko']
  ➗🇱🇻 v14 w14 mapping: split 'tu esi vesels' into ['tu', 'esi', 'vesels']
  ➗🇱🇻 v14 w17 mapping: split 'vairs lai tev nenotiek' into ['vai

In [140]:
from datetime import datetime
fix_frame.to_csv(f"autofix_log_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv", index=False)

In [123]:
lv65_df.query("book == '1_corinthians' and chapter == 7 and verse == 5")['text'].astype(str).iloc[0]

'Neatraujieties viens otram kā vien uz kādu laiku pēc pašu vienošanās lai atrastu laiku Dieva lūgšanai un tad turieties atkal kopā lai sātans jūs nekārdinātu jūsu nesavaldības dēļ'

## 5 – Fix summary

In [104]:
fix_types = Counter(f[0] for f in all_fixes)
print("Fix breakdown:")
for t, c in fix_types.most_common():
    print(f"  {t}: {c}")

# Per-book summary
book_fix_counts = Counter() 
for f in all_fixes:
    if len(f) >= 2 and isinstance(f[1], tuple) and len(f[1]) >= 1:
        book_fix_counts[f[1][0]] += 1

print(f"\nPer-book fix counts:")
for book, count in book_fix_counts.most_common(): 
    print(f"  {book}: {count}")

Fix breakdown:
  greek_clean: 135245
  greek_extra_removed: 2
  greek_missing_inserted: 2

Per-book fix counts:
  luke: 19144
  matthew: 18112
  acts: 18044
  john: 15293
  mark: 11219
  revelation: 9696
  romans: 7025
  1_corinthians: 6230
  hebrews: 4841
  2_corinthians: 4371
  ephesians: 2404
  galatians: 2189
  1_john: 2157
  james: 1706
  1_peter: 1678
  philippians: 1583
  1_timothy: 1567
  colossians: 1559
  1_thessalonians: 1479
  2_timothy: 1192
  2_peter: 1088
  2_thessalonians: 812
  titus: 647
  jude: 458
  philemon: 309
  2_john: 238
  3_john: 208


## 6 – (Optional) Run on a single book/chapter for testing

In [ ]:
# Test on a single chapter first
# had, log = autofix_chapter('acts', 11, strongs_g, lv_g, verbose=True, dry_run=True)
# print(f"\nFixes: {len(log)}")